# Lab 2: Self-Attention do Zero

## Lab 2: Self-Attention calculada do zero

In [1]:
!pip install -q torch transformers matplotlib

import matplotlib
matplotlib.use("Agg")  # backend não-interativo — salva em arquivo, não abre janela

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

### 1. Scaled dot-product attention — implementação manual

**Por que fazer isso na mão:** ver a fórmula funcionar em 5 linhas de
PyTorch tira o mistério — não tem nada escondido, é álgebra linear.

In [2]:
def scaled_dot_product_attention(Q, K, V, causal_mask=True):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)

    if causal_mask:
        seq_len = scores.shape[-1]
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))

    attention_weights = F.softmax(scores, dim=-1)
    output = attention_weights @ V
    return output, attention_weights

# Sequência de 6 tokens, embedding de dimensão 8 (números pequenos pra caber na tela)
torch.manual_seed(42)
seq_len, d_model = 6, 8
x = torch.randn(seq_len, d_model)

# Q, K, V — normalmente vêm de matrizes aprendidas; aqui usamos projeções lineares simples
W_q, W_k, W_v = torch.randn(d_model, d_model), torch.randn(d_model, d_model), torch.randn(d_model, d_model)
Q, K, V = x @ W_q, x @ W_k, x @ W_v

output, weights = scaled_dot_product_attention(Q, K, V, causal_mask=True)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nMatriz de attention weights (6x6, cada linha soma 1):")
print(weights.round(decimals=3))

Input shape: torch.Size([6, 8])
Output shape: torch.Size([6, 8])

Matriz de attention weights (6x6, cada linha soma 1):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0010, 0.9990, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.9800, 0.0200, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000]])


**Resultado esperado:** a matriz de weights é 6x6, triangular (a máscara
causal zera a metade superior — token 0 só olha pra si mesmo, token 5 olha
pra todos os 6). Cada linha soma exatamente 1.0 (é uma distribuição de
probabilidade, garantida pelo softmax).

### 2. Visualizando a máscara causal

In [3]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights.detach().numpy(), cmap="Blues")
ax.set_xlabel("Key position (o que está sendo 'olhado')")
ax.set_ylabel("Query position (quem está 'olhando')")
ax.set_title("Attention weights (triangular = causal mask)")
plt.colorbar(im)
plt.savefig("attention_heatmap.png", dpi=80, bbox_inches="tight")
plt.close(fig)
print("✓ Heatmap salvo em attention_heatmap.png")

✓ Heatmap salvo em attention_heatmap.png


**Resultado esperado:** um heatmap triangular — a metade superior direita
(tokens futuros) fica em branco/zero, confirmando visualmente a máscara
causal da Semana 2.5.

### 3. Comparando com a implementação real do PyTorch

**Por que importa:** confirma que nossa implementação manual bate com a
função otimizada que a `transformers`/PyTorch realmente usa em produção
(`scaled_dot_product_attention` é uma função nativa do PyTorch desde a
versão 2.0, com kernels otimizados — a matemática é a mesma que escrevemos
à mão).

In [4]:
output_native, weights_native = F.scaled_dot_product_attention(
    Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0),
    is_causal=True,
), None

# A função nativa não retorna os weights por padrão — comparamos só o output
diff = (output_native.squeeze(0) - output).abs().max().item()
print(f"Diferença máxima entre implementação manual e nativa: {diff:.2e}")
assert diff < 1e-5, "As implementações deveriam ser numericamente equivalentes!"
print("✓ Implementação manual bate com a função nativa do PyTorch")

Diferença máxima entre implementação manual e nativa: 4.77e-07
✓ Implementação manual bate com a função nativa do PyTorch


**Resultado esperado:** `Diferença máxima: ~1e-07` (praticamente zero — a
diferença vem só de arredondamento de ponto flutuante) e a confirmação
`✓ Implementação manual bate com a função nativa`.

### 4. Multi-head attention — múltiplas "cabeças" em paralelo

In [5]:
def multi_head_attention(x, n_heads=2):
    seq_len, d_model = x.shape
    d_head = d_model // n_heads
    assert d_model % n_heads == 0

    outputs = []
    for h in range(n_heads):
        torch.manual_seed(h)  # cada cabeça tem pesos diferentes
        W_q, W_k, W_v = torch.randn(d_model, d_head), torch.randn(d_model, d_head), torch.randn(d_model, d_head)
        Q, K, V = x @ W_q, x @ W_k, x @ W_v
        head_out, _ = scaled_dot_product_attention(Q, K, V, causal_mask=True)
        outputs.append(head_out)

    return torch.cat(outputs, dim=-1)  # concatena as cabeças

multi_out = multi_head_attention(x, n_heads=2)
print(f"Multi-head output shape: {multi_out.shape}")  # (6, 8) — mesma dimensão do input, 2 cabeças de 4

Multi-head output shape: torch.Size([6, 8])


**Resultado esperado:** `Multi-head output shape: torch.Size([6, 8])` — 2
cabeças de dimensão 4 cada, concatenadas de volta pra dimensão 8 (o mesmo
`d_model` do input), exatamente como descrito na Semana 2.5.

**Próximos passos:** Semana 3 mostra como os pesos das matrizes Q/K/V
(que aqui inicializamos aleatoriamente) são de fato **aprendidos** via
gradient descent.